# Module 3 - Distillation: teach a small model to reason from a large one's work

Modules 2a and 2b used a large model to read papers. Here we do the opposite direction of value transfer: we take a large model's *reasoning* and press it into a small model you can host yourself.

The task is chemical. Given a compound and two candidate solvents, decide which solvent dissolves it better at a given temperature (solubench, task 1). The dataset ships only the answer, A or B, never the why. A small model trained on answers alone barely clears a coin flip on held-out compounds, because the label carries no chemistry.

So we bring in a large teacher, DeepSeek v4-flash, to write a short step-by-step reason for each training compound, and we fine-tune a small student, Qwen3-1.7B, on those reasons with LoRA. Then we measure the student before and after. This is knowledge distillation: the teacher's answer is cheap, its reasoning is the thing worth copying.

**Two tracks, one switch `USE_CACHE`:**

| track | needs | what runs |
|---|---|---|
| cache (default) | nothing, no GPU, no key | reads the shipped teacher chains and the before/after results, reproduces the whole story offline |
| live | a Colab **T4 GPU**; a DeepSeek key only if you regenerate chains | actually generates, trains with LoRA, and evaluates |

To run live, open this notebook in Colab and set the runtime to a T4 GPU (Runtime, Change runtime type, T4), then set `USE_CACHE = False` in the config cell.

## 0. Setup

In [ ]:
# Colab setup: fetch the data and enter this module's folder. No-op when run locally.
import os, sys
REPO = "https://github.com/ruiding-uchicago/NRT_Training_Materials.git"
MODULE = "module_3_distillation"
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != MODULE:
    root = "/content/NRT_Training_Materials"
    if not os.path.isdir(root):
        os.system("git clone --depth 1 " + REPO + " " + root)
    os.chdir(root + "/" + MODULE)
print("ready in", os.getcwd())

In [ ]:
# Config. Flip USE_CACHE to False on a Colab T4 to train and evaluate for real.
USE_CACHE = True

STUDENT  = "Qwen/Qwen3-1.7B"     # the small student we fine-tune
N_EVAL   = 200                   # held-out compounds scored before and after (raise to 500 for the full set)
EPOCHS   = 2
LORA_R, LORA_ALPHA = 16, 32
SEED = 42

import os
DATA = "data"; ADAPT = "adapter"; os.makedirs(ADAPT, exist_ok=True)
CHAINS = f"{DATA}/chains_task1_train.jsonl"   # teacher reasoning, filtered to correct
TEST   = f"{DATA}/task1_test.jsonl"           # held-out compounds, answer only
print("cache mode" if USE_CACHE else "live mode (needs a T4 GPU)")

In [ ]:
# Live mode installs the training stack. No-op in cache mode or when already present.
import sys, subprocess
if not USE_CACHE and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-U", "transformers", "trl", "peft", "accelerate", "datasets"])

## 1. The data: answers without reasons

Each row is one compound and two solvents; the label is a single letter, A or B, for which solvent dissolves it better. That is all solubench gives us. The chemistry that connects a structure to a solvent choice, polarity, hydrogen bonding, chain length, is left implicit. That gap is exactly what a small model cannot fill from labels alone, and exactly what the teacher will make explicit.

In [ ]:
import json
def load_jsonl(p):
    txt = open(p).read().strip()
    return json.loads(txt) if txt[:1] == "[" else [json.loads(l) for l in txt.splitlines() if l.strip()]

test   = load_jsonl(TEST)
chains = load_jsonl(CHAINS)
print(f"held-out test compounds: {len(test)}")
print(f"teacher reasoning chains (train): {len(chains)}")
r = test[0]
print("\na held-out item, answer only:")
print(f"  compound {r['smiles']}  in  A={r['sa']}  vs  B={r['sb']}  at {r['T']} K   ->  gold {r['gold']}")

In [ ]:
# one full teacher chain: structure -> polarity and H-bonding -> a final choice
c = chains[0]
print("teacher reasoning chain  (gold answer:", c["gold"], ")")
print("-" * 70)
print(c["content"])

## 2. The teacher writes the reasoning, and the 16-point trap

The teacher sees the compound, the two solvents, and the verified answer, and is asked to reason first and then commit to a final line. One detail decides whether the chains are worth training on: **how you hand the teacher the answer.**

If you prompt the teacher to *write the derivation toward this answer*, it reasons backward from the label. The chain looks fluent but stops being a real derivation, and a student trained on it learns to assert rather than reason. We measured this on our own large-model run: that phrasing cost **16 points on MATH-500** against the alternative of giving the answer only as a final check. So we use the careful phrasing, the answer is for verification of the last line, not a premise to restate.

In [ ]:
# The two prompts. We ship chains built with the good one. The difference is the whole lesson.
GOOD = """... Reason step by step from the compound's structure and each solvent's polarity and
hydrogen-bonding capacity, then end with your final choice on its own line as "Answer: A" or "Answer: B".
(For checking only, the verified answer is {gold}. Use it to verify your final line; do not restate it
as given, and do not skip the reasoning.)"""

BAD  = """... The answer is {gold}. Write the derivation that leads to it."""   # do not use: teaches assertion

print("good (used):", GOOD.strip()[-180:])
print("\nbad  (avoided):", BAD.strip())

In [ ]:
# Live regeneration is optional and needs a DeepSeek key. Cache mode uses the shipped chains.
# Kept here so the pipeline is visible end to end. Key: thinking is disabled (else ~3x cost from
# billed reasoning tokens), read the plain content, high worker count for throughput.
DEEPSEEK_KEY = ""   # paste sk-... to regenerate; leave blank to use shipped chains

PROMPT = """A compound is tested in two candidate solvents. Decide which solvent dissolves it better at the given temperature.

Compound (SMILES): {smiles}
Temperature: {T} K
Solvent A: {sa}
Solvent B: {sb}

Reason step by step from the compound's structure and each solvent's polarity and hydrogen-bonding capacity, then end with your final choice on its own line as "Answer: A" or "Answer: B".
(For checking only, the verified answer is {gold}. Use it to verify your final line; do not restate it as given, and do not skip the reasoning.)"""

def teacher_call(smiles, T, sa, sb, gold, key):
    import requests
    body = {"model": "deepseek-v4-flash",
            "messages": [{"role": "user", "content": PROMPT.format(smiles=smiles, T=T, sa=sa, sb=sb, gold=gold)}],
            "temperature": 0.7, "thinking": {"type": "disabled"}, "max_tokens": 2000}
    r = requests.post("https://api.deepseek.com/chat/completions",
                      headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
                      data=json.dumps(body), timeout=120)
    return r.json()["choices"][0]["message"]["content"]

if not USE_CACHE and DEEPSEEK_KEY:
    demo = teacher_call(test[0]["smiles"], test[0]["T"], test[0]["sa"], test[0]["sb"], test[0]["gold"], DEEPSEEK_KEY)
    print("fresh teacher chain for one held-out compound:\n"); print(demo)
else:
    print("using shipped chains (", len(chains), "of them ). Set DEEPSEEK_KEY and USE_CACHE=False to regenerate.")

## 3. Build the training set

We turn each teacher chain into one chat example: the clean question as the user turn, the reasoning ending in `Answer: X` as the assistant turn. Two rules keep it honest. The answer hint that the teacher saw is stripped, so the student never reads the label in its prompt. And we keep only chains whose final line matches the gold answer, so the student trains on correct reasoning, not on the teacher's rare misses.

In [ ]:
import re
QUESTION = """A compound is tested in two candidate solvents. Decide which solvent dissolves it better at the given temperature.

Compound (SMILES): {smiles}
Temperature: {T} K
Solvent A: {sa}
Solvent B: {sb}

Reason step by step from the compound's structure and each solvent's polarity and hydrogen-bonding capacity, then end with your final choice on its own line as "Answer: A" or "Answer: B"."""

def strip_hint(q):
    return re.sub(r"\n*\(For checking only.*", "", q, flags=re.S).strip()

sft = []
for c in chains:
    ans = re.findall(r"Answer:\s*([AB])\b", c["content"])
    if not ans or ans[-1] != c["gold"]:
        continue
    user = strip_hint(c["question"]) if "question" in c else QUESTION.format(**c)
    sft.append({"messages": [{"role": "user", "content": user},
                             {"role": "assistant", "content": c["content"].strip()}]})

print(f"training examples: {len(sft)}")
print("no answer hint leaked into any prompt:", not any("For checking" in e["messages"][0]["content"] for e in sft))
print("\nassistant turn ends on:", repr(sft[0]["messages"][1]["content"][-24:]))

## 4. The student before training

We load Qwen3-1.7B in its non-thinking mode, give it the same question, decode greedily, and read the `Answer` line. This is the baseline: what the small model knows about solvent selection before it sees any of the teacher's reasoning.

In [ ]:
import json
def qtext(r):
    return QUESTION.format(smiles=r["smiles"], T=r["T"], sa=r["sa"], sb=r["sb"])

def evaluate(model, tok, items, max_new=384, batch=16):
    import torch
    dev = next(model.parameters()).device
    preds = []
    for i in range(0, len(items), batch):
        chunk = items[i:i+batch]
        texts = [tok.apply_chat_template([{"role": "user", "content": qtext(r)}],
                 tokenize=False, add_generation_prompt=True, enable_thinking=False) for r in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, padding_side="left").to(dev)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for j in range(len(chunk)):
            gen = tok.decode(out[j, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            m = re.findall(r"Answer:\s*([AB])\b", gen)
            preds.append((m[-1] if m else "?", gen))
    acc = sum(p == r["gold"] for (p, _), r in zip(preds, items)) / len(items)
    return acc, preds

if not USE_CACHE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(STUDENT)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=torch.float16, device_map="cuda")
    base.eval()
    ev = test[:N_EVAL]
    acc_before, preds_before = evaluate(base, tok, ev)
    json.dump({"acc": acc_before, "n": len(ev)}, open(f"{DATA}/eval_before.json", "w"))
    print(f"before: {acc_before:.3f} on {len(ev)} compounds")
else:
    acc_before = json.load(open(f"{DATA}/eval_before.json"))["acc"]
    print(f"before (shipped): {acc_before:.3f}")

## 5. LoRA fine-tune on the teacher's reasoning

LoRA freezes the 1.7B base and trains a small pair of low-rank matrices on top, so a T4 is enough and the result is a lightweight adapter, not a new copy of the model. We train for two passes over the reasoning chains.

In [ ]:
if not USE_CACHE:
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig

    ds = Dataset.from_list(sft)
    peft_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
                          task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                          "gate_proj", "up_proj", "down_proj"])
    args = SFTConfig(output_dir="_train", num_train_epochs=EPOCHS,
                     per_device_train_batch_size=8, gradient_accumulation_steps=2,
                     learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type="cosine",
                     logging_steps=20, save_strategy="no", bf16=False, fp16=True,
                     max_length=1024, seed=SEED, report_to=[])
    trainer = SFTTrainer(model=base, args=args, train_dataset=ds,
                         processing_class=tok, peft_config=peft_cfg)
    trainer.train()
    trainer.model.save_pretrained(ADAPT)
    tok.save_pretrained(ADAPT)
    print("adapter saved to", ADAPT)
else:
    print("using the shipped adapter in", ADAPT)

## 6. The student after training

Same compounds, same decoding, now with the adapter attached. We score the held-out set again and keep a few before/after generations to read.

In [ ]:
if not USE_CACHE:
    tuned = trainer.model.eval()          # already carries the trained LoRA
    acc_after, preds_after = evaluate(tuned, tok, ev)
    json.dump({"acc": acc_after, "n": len(ev)}, open(f"{DATA}/eval_after.json", "w"))
    samples = [{"compound": r["smiles"], "gold": r["gold"],
                "before": preds_before[i][1], "after": preds_after[i][1]}
               for i, r in enumerate(ev[:5])]
    json.dump(samples, open(f"{DATA}/samples.json", "w"), ensure_ascii=False, indent=1)
    print(f"after: {acc_after:.3f} on {len(ev)} compounds")
else:
    acc_after = json.load(open(f"{DATA}/eval_after.json"))["acc"]
    print(f"after (shipped): {acc_after:.3f}")

## 7. Before vs after

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.6, 4.2))
bars = ax.bar(["before", "after"], [acc_before, acc_after], color=["#b0b0b0", "#5b8c5a"], width=0.6)
ax.set_ylim(0, 1); ax.set_ylabel("accuracy on held-out compounds")
ax.set_title("Distilling the teacher's reasoning into a 1.7B student")
for b, v in zip(bars, [acc_before, acc_after]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")
ax.axhline(0.5, ls="--", lw=1, color="#c0392b"); ax.text(1.4, 0.51, "coin flip", color="#c0392b", fontsize=8)
plt.tight_layout(); os.makedirs("figures", exist_ok=True); plt.savefig("figures/before_after.png", dpi=140); plt.show()
print(f"lift: {acc_before:.3f} -> {acc_after:.3f}  (+{acc_after-acc_before:.3f})")

In [ ]:
# read one compound the way the student answers it, before and after
import json, os
if os.path.exists(f"{DATA}/samples.json"):
    s = json.load(open(f"{DATA}/samples.json"))[0]
    print("compound:", s["compound"], "| gold:", s["gold"])
    print("\n--- before ---\n", s["before"].strip()[:500])
    print("\n--- after ---\n",  s["after"].strip()[:500])

## 8. What this bought, and how to use it on your data

- A dataset that ships only answers becomes reasoning supervision a small model can absorb. The label told the student nothing; the teacher's chain taught it the chemistry that maps a structure to a solvent.
- The asset is the teacher's reasoning, not its answer. We trained only on chains whose final line was correct, and stripped the answer from every prompt, so the student learned to derive rather than to recall.
- The 16-point lesson stands on its own: how you present the answer to the teacher decides whether the chains are derivations or after-the-fact assertions. Give the answer as a final check, never as a premise to restate.

To use this on your own field, replace solubench task 1 with any answer-only dataset in the same shape, one question and a short verifiable label. Regenerate the chains with the teacher, rebuild the chat set, and retrain. The pipeline does not change.